In [ ]:
# ✅ Step 1: Install Required Libraries
%pip install tensorflow matplotlib scikit-learn


In [2]:
# ✅ Step 2: Import Libraries
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# ✅ Step 3: Set Dataset Path
dataset_path = r"C:\Users\alluv\Transforming-Waste-Management-with-Transfer-Learning\Project files\garbage_classification"

In [4]:
# ✅ Step 4: Load & Preprocess Dataset
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 5762 files belonging to 12 classes.
Using 4610 files for training.
Found 5762 files belonging to 12 classes.
Using 1152 files for validation.
Classes: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'footwear', 'green-glass', 'metal', 'paper', 'plastic', 'trash', 'white-glass']


In [5]:
# ✅ Step 5: Prefetch for Performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [6]:
# ✅ Step 6: Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),  
    tf.keras.layers.RandomZoom(0.2),      
    tf.keras.layers.RandomContrast(0.2),  
    tf.keras.layers.RandomBrightness(0.2),
])

In [7]:
# ✅ Step 7: Load MobileNetV2 base model
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False

In [8]:
# ✅ Step 8: Create full model
model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

In [9]:
# ✅ Step 9: Compile model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
# ✅ Step 10: Train initial model

# Add this to your training code
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "waste_classifier_checkpoint.weights.h5", save_best_only=False, save_weights_only=True
)

initial_epochs = 15 
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epochs,
    callbacks=[checkpoint_cb]
)

# To resume after shutdown
# model.load_weights("waste_classifier_checkpoint.h5")
# Then call model.fit() again with initial_epoch set to the last completed epoch


Epoch 1/15
 87/145 ━━━━━━━━━━━━━━━━━━━━ 39s 686ms/step - accuracy: 0.2433 - loss: 2.2926

In [ ]:
# ✅ Step 11: Fine-tune model

base_model.trainable = True
fine_tune_at = 25  
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-6),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_epochs = 10  
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(train_ds,
                         validation_data=val_ds,
                         epochs=total_epochs,
                         initial_epoch=history.epoch[-1])


In [ ]:
def predict_image(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) / 255.0
    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions)
    print(f"Predicted: {predicted_class} ({confidence:.2f})")


In [ ]:
# Evaluate the model on the validation dataset and print the accuracy
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# ✅ Step 14: Save the trained model
model.save("waste_classifier_model.h5")
